# 02 — TRIBE inference over the stimulus corpus

Track A, Colab GPU. Requires notebook 01 to have passed gate 17 for the pinned
revision.

> ## ⚠️ Commit results before the session closes
>
> Colab sessions die at 12h, on disconnect, and on idle. Predictions are
> written to **Drive** inside the loop, so they survive — but the manifest
> summary and any repo-side artifact do not. Before you close this notebook:
>
> 1. run the manifest verification cell,
> 2. commit and push any repo changes from a machine with push access.
>
> The run is idempotent: a stimulus already in the cache is skipped, never
> regenerated (§4.3). Re-running after a dropped session resumes; it does not
> restart.

Nothing below computes anything itself.

## Bootstrap

1. **GPU check** — Track A needs one. Runtime → Change runtime type → GPU.
2. **Clone `phase2-tribe` and install**, then **Runtime → Restart session**.
   The branch is not optional: the repo's default branch has no `src/tribe/`.
3. **Session setup** — mount Drive, derive the cache paths, set `HF_HOME`.
4. **Authenticate** from Colab Secrets (🔑), never from a literal in a cell.

Steps 3 and 4 re-derive everything they need, so they are what you re-run after
a restart or a dropped connection — nothing above them is needed again.

`HF_HOME` must be set *before* any HuggingFace import in the process. Once a HF
module is imported the cache location is fixed, and several GB of checkpoint
land on the ephemeral runtime disk instead of Drive.

Everything after that is a single call into a script in `scripts/`. No project
logic lives in this notebook: a cell dies with the session, and brief §4.3
requires every result to come from a script in the repo.

In [ ]:
# 1. GPU check
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU")

In [ ]:
# 2. Clone the repo and install the pinned Track A stack into Colab's interpreter
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"
# Not optional. Phase II lives on `phase2-tribe`; the default branch (`master`)
# has no src/tribe/ and no scripts/run_tribe_*.py, so an unpinned clone fails
# further down with a confusing "No such file or directory".
BRANCH = "phase2-tribe"
REPO_DIR = Path("/content/neurotutorsim")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
# --system installs into Colab's own interpreter. `uv sync` would build a venv
# this kernel cannot import from.
#
# `.[tribe,dev]`, NOT `--extra tribe --extra dev`: in pip mode uv rejects --extra
# unless the target is `-r <file>`. With `-e .` it exits 2 with
#   "Requesting extras requires a ... pyproject.toml ... Use <dir>[extra] instead"
subprocess.run(
    ["uv", "pip", "install", "--system", "-e", ".[tribe,dev]"],
    cwd=REPO_DIR,
    check=True,
)

print("installed; Runtime -> Restart session, then run the cell below")

### ⚠️ Runtime → Restart session now

Then continue from the cell below. It re-derives every path it needs, so
nothing above has to run again — and it is also the first cell to re-run after
a disconnect.

In [ ]:
# 3. Session setup: Drive, cache paths, HF_HOME. Self-contained on purpose --
#    this is the cell to run first after a restart or a dropped connection.
import os
import sys
from pathlib import Path

# Before any HuggingFace import in this process. Set afterwards it does nothing:
# the cache location is fixed at import time, and the checkpoint would land on
# the ephemeral runtime disk to be re-downloaded every session.
assert not any(m.startswith(("huggingface_hub", "transformers")) for m in sys.modules), (
    "a HuggingFace module is already imported; Runtime -> Restart session and run this cell first"
)

try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
except ModuleNotFoundError:
    DRIVE_ROOT = Path("./drive_local")   # off Colab: keep the notebook runnable

# Storage is the Drive of whichever account is signed into Colab -- 15 GB on a
# personal account, Pro included. `drive.mount` cannot reach a second account, so
# moving to a larger Drive means running Colab from that account, not editing a
# path here. Full-corpus estimate: ~1.2 GB parcel + ~3.5 GB vertex + ~7 GB HF.
PROJECT_DRIVE = DRIVE_ROOT / "NeuroTutorSim"
CACHE_ROOT = PROJECT_DRIVE / "tribe_cache"
HF_CACHE = PROJECT_DRIVE / "hf"
HANDOFF = PROJECT_DRIVE / "gate17"          # written by notebook 01, read by 02
REPO_DIR = Path("/content/neurotutorsim")
for d in (PROJECT_DRIVE, CACHE_ROOT, HF_CACHE, HANDOFF):
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
# Exported, not just assigned: the `!` cells below launch a subprocess, which
# inherits the environment but knows nothing about this kernel's variables.
os.environ["CACHE_ROOT"] = str(CACHE_ROOT)

print(f"repo       : {REPO_DIR}  (exists={REPO_DIR.exists()})")
print(f"cache root : {CACHE_ROOT}")
print(f"HF cache   : {HF_CACHE}")

In [ ]:
# 4. Authenticate. HF_TOKEN comes from Colab Secrets (the key icon), never a literal.
#    `meta-llama/Llama-3.2-3B` (TRIBE's text encoder) is gated per account: a read
#    token grants nothing until Meta approves the request for that account. Colab
#    Secrets are per Google account too, so a new account needs HF_TOKEN added
#    again with notebook access toggled on.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
print("authenticated")

## Preflight

Two things live outside the clone and have to be put back before the loop can
start:

* **`outputs/verification/gate17.json`** — `outputs/` is gitignored, so a fresh
  clone never has it. Restored from the Drive handoff written by notebook 01.
* **the pinned revision** — if `config/tribe.yaml` and `config/checkpoints.lock`
  have not been committed yet, the clone still carries the all-zero placeholder
  and the gate artifact will not match. Restored from the same handoff.

The Schaefer-400 atlas is *not* on this list: `schaefer400_fsaverage5.npz` is
committed to the repo (21 KB), so the clone already has it. The cell checks it
anyway — `run_inference` loads it before the first GPU call.

In [ ]:
# Preflight. Nothing here computes: it restores state and reports readiness.
import shutil
import subprocess
import sys
from pathlib import Path

# 1. gate 17 artifact, back into the path inference.py looks at
artifact = REPO_DIR / "outputs" / "verification" / "gate17.json"
stored = HANDOFF / "gate17.json"
if artifact.exists():
    print(f"gate17.json  : already in the clone")
elif stored.exists():
    artifact.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(stored, artifact)
    print(f"gate17.json  : restored from {stored}")
else:
    print(f"gate17.json  : NOT FOUND (looked in {stored}) -- run notebook 01 first")

# 2. the pin. Only used if the committed config is still the placeholder, so a
#    properly committed pin always wins over a stale Drive copy.
sys.path.insert(0, str(REPO_DIR))
from src.tribe.config import UNRESOLVED_REVISION, load_config

config_path = REPO_DIR / "config" / "tribe.yaml"
if load_config(config_path).checkpoint_revision == UNRESOLVED_REVISION:
    for name in ("tribe.yaml", "checkpoints.lock"):
        source = HANDOFF / name
        if source.exists():
            shutil.copy2(source, REPO_DIR / "config" / name)
            print(f"config/{name} : restored from Drive (not yet committed)")
        else:
            print(f"config/{name} : MISSING from {HANDOFF} and unpinned in git")
else:
    print("revision     : pinned in the committed config")

# 3. atlas
atlas = REPO_DIR / "data" / "raw" / "atlas" / "schaefer400_fsaverage5.npz"
print(
    f"atlas        : {atlas.stat().st_size:,} B"
    if atlas.exists()
    else "atlas        : MISSING -- run notebook 03 and commit the .npz"
)

# 4. the gate, read exactly as run_tribe_inference.py reads it
print()
subprocess.run(
    [
        sys.executable,
        "-c",
        "from src.tribe.config import load_config; from src.tribe import verification; "
        "c = load_config('config/tribe.yaml'); "
        "print('gate_17_passed =', verification.gate_17_passed(c))",
    ],
    cwd=REPO_DIR,
    check=True,
)

## Run the corpus

Parcel output for every stimulus, vertex output only for the ids in
`config/vertex_retention_set.txt`. Both are written to Drive inside the loop, so
a session that dies loses at most the stimulus in flight.

In [ ]:
!cd /content/neurotutorsim && python scripts/run_tribe_inference.py \
    --config config/tribe.yaml --cache-root "$CACHE_ROOT"

## Optional: determinism check (§10.1)

Runs one stimulus twice and requires **bitwise-equal** output. Not a tolerance
check — if it fails, stop and report it rather than relaxing the test.

In [ ]:
!cd /content/neurotutorsim && python scripts/run_tribe_inference.py \
    --config config/tribe.yaml --cache-root "$CACHE_ROOT" --check-determinism

## Before closing: verify the manifest

Reports entries recorded but missing on disk, and files on disk missing from
the manifest. It deletes nothing — every entry cost a GPU run against a gated
model.

In [ ]:
!cd /content/neurotutorsim && python scripts/run_tribe_inference.py \
    --config config/tribe.yaml --cache-root "$CACHE_ROOT" --verify-manifest

## What is on Drive now

Under `MyDrive/NeuroTutorSim/tribe_cache/`:

| | path | size |
|---|---|---|
| parcel | `parcel/<first2>/<parcel_key>.parquet` | ~200 KB per stimulus |
| vertex | `vertex/<first2>/<vertex_key>.npz` | ~4 MB, retention set only |
| index | `manifest.parquet` | one row per entry |

Filenames are content hashes, not stimulus ids — `manifest.parquet` is the only
map from `be_001_traditional_primary` to a file on disk. Track B reads the
parcel tables through `TribeCache.resolve`, which reconstructs the key from the
stimulus text and config, so the hashes never have to be handled by hand.

In [ ]:
import pandas as pd

manifest = pd.read_parquet(CACHE_ROOT / "manifest.parquet")
print(f"{len(manifest)} entries in {CACHE_ROOT / 'manifest.parquet'}\n")
print(
    manifest[["stimulus_id", "level", "n_timesteps", "size_bytes", "path"]]
    .to_string(index=False)
)